# Quantum phase estimation, on qBraid and on IBM

| | |
|---|---|
| **Level** | Intermediate |
| **Time** | About 60 minutes |
| **Prerequisites** | Controlled gates, the quantum Fourier transform (helpful) |
| **Default device** | IQM Garnet through qBraid |
| **Also runs on** | Rigetti Cepheus-1-108Q through qBraid, and IBM Quantum devices with your own IBM account |
| **Qubits** | 4 |
| **Two-qubit gates** | about 15 before routing, about 20 after routing on a square lattice |
| **Hardware jobs** | 1 on qBraid, plus 1 on IBM if you choose |
| **Approximate cost** | Garnet at 500 shots: about 103 credits. IBM runs use your IBM allocation, not qBraid credits. |
| **Suggested hand-in** | Your comparison chart, and answers to Questions 1 and 2 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST notebooks from qBraid. You may copy, edit and adapt this notebook for your course.*

**Quantum phase estimation (QPE)** finds the eigenvalue of a unitary. If $U|\psi\rangle = e^{2\pi i \varphi}|\psi\rangle$, QPE writes the phase $\varphi$ into a register of counting qubits, one binary digit per qubit. It is a building block of Shor's algorithm and of many chemistry algorithms.

Here $U$ is the phase gate $P(2\pi\varphi)$ and $|\psi\rangle = |1\rangle$, so the correct answer is known in advance. With three counting qubits and $\varphi = 3/8$, the output should be the binary number `011`.

This notebook also shows how to run on **IBM Quantum** devices from qBraid Lab. IBM's devices are not paid for with qBraid credits; you need your own IBM Quantum account, and the IBM section is skipped unless you turn it on.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "aws:iqm:qpu:garnet"   # device list and prices: see the README
SHOTS = 500                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits
USE_IBM = False             # set to True to also run on an IBM device (needs your IBM account, see section 4)
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']   # gates every QUEST device accepts
QUEST_JOB_TAGS = {"quest": "algo-qpe"}   # labels this notebook's hardware jobs for QUEST usage statistics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuit

Counting qubit $j$ controls $U^{2^j}$. The inverse quantum Fourier transform then turns the collected phases into a binary number.

In [ ]:
PHASE = 3 / 8        # the phase to estimate. 3/8 = 0.011 in binary, so the answer is exact.
N_COUNT = 3          # counting qubits: the answer has this many binary digits

def inverse_qft(qc, n):
    for j in range(n // 2):
        qc.swap(j, n - j - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)

def qpe_circuit(phase, n_count):
    qc = QuantumCircuit(n_count + 1, n_count)
    target = n_count
    qc.x(target)                                   # the eigenstate |1> of the phase gate
    qc.h(range(n_count))
    for j in range(n_count):
        qc.cp(2 * np.pi * phase * 2 ** j, j, target)   # controlled U^(2^j)
    inverse_qft(qc, n_count)
    qc.measure(range(n_count), range(n_count))
    return qc

qc = qpe_circuit(PHASE, N_COUNT)
answer = format(round(PHASE * 2 ** N_COUNT) % 2 ** N_COUNT, f"0{N_COUNT}b")
print("expected answer:", answer)
qc.draw(output="text", fold=120)

## 2. Ideal simulation

In [ ]:
simulator = AerSimulator()
ideal_counts = simulator.run(qc, shots=SHOTS).result().get_counts()
print(ideal_counts)

def p_answer(counts):
    total = sum(counts.values())
    return sum(n for key, n in counts.items() if key.replace(" ", "").zfill(N_COUNT) == answer) / total

## 3. Run on a qBraid device

In [ ]:
qc_hw = transpile(qc, basis_gates=HW_BASIS, optimization_level=1)
ops = qc_hw.count_ops()
print(f"two-qubit gates before routing: {ops.get('cx', 0) + ops.get('cz', 0)}")

In [ ]:
N_JOBS = 1

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    job = device.run(qc_hw, shots=SHOTS, tags=QUEST_JOB_TAGS)
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = job.result().data.get_counts()
    print(f"P(correct answer) = {p_answer(hw_counts):.3f}")
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Run on an IBM device

IBM runs its own cloud service. From qBraid Lab you can reach it with the `qiskit-ibm-runtime` package and your own IBM credentials. IBM's free plan includes a limited amount of device time each month; check IBM's current terms.

**One-time setup:**

1. Create an account at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com).
2. Create an instance on the free plan.
3. Copy your **API key** and your instance's **CRN**. Treat the API key like a password.
4. If your environment does not have it, install the package with `%pip install qiskit-ibm-runtime` and restart the kernel.
5. Set `USE_IBM = True` in Settings and run the next two cells. The first asks for your key once and saves it in your qBraid Lab home folder, so you do not need to paste it into the notebook.

In [ ]:
ibm_counts = None
if USE_IBM:
    from getpass import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService
    if not QiskitRuntimeService.saved_accounts():
        QiskitRuntimeService.save_account(
            channel="ibm_quantum_platform",
            token=getpass("IBM API key: "),
            instance=input("Instance CRN: "),
            set_as_default=True,
        )
    service = QiskitRuntimeService()
    ibm_backend = service.least_busy(operational=True, simulator=False)
    print("Least busy IBM device:", ibm_backend.name)
else:
    print("IBM section skipped. Set USE_IBM = True in Settings to run it.")

In [ ]:
if USE_IBM:
    from qiskit_ibm_runtime import SamplerV2 as Sampler
    qc_ibm = transpile(qc, backend=ibm_backend, optimization_level=2)   # IBM's own gates and layout
    ops = qc_ibm.count_ops()
    print(f"two-qubit gates on {ibm_backend.name}: {ops.get('cz', 0) + ops.get('ecr', 0) + ops.get('cx', 0)}")
    ibm_job = Sampler(mode=ibm_backend).run([qc_ibm], shots=SHOTS)
    print("Submitted to IBM. Job ID:", ibm_job.job_id())
    ibm_counts = ibm_job.result()[0].data.c.get_counts()
    print(f"P(correct answer) = {p_answer(ibm_counts):.3f}")

## 5. Compare

In [ ]:
labels, values, colors = ["ideal simulation"], [p_answer(ideal_counts)], ["gray"]
if hw_counts:
    labels.append(f"qBraid: {DEVICE_ID.split(':')[-1]}")
    values.append(p_answer(hw_counts))
    colors.append("tab:orange")
if ibm_counts:
    labels.append(f"IBM: {ibm_backend.name}")
    values.append(p_answer(ibm_counts))
    colors.append("tab:blue")

plt.bar(labels, values, color=colors)
plt.axhline(1 / 2 ** N_COUNT, color="black", linestyle=":", label=f"random guessing (1/{2 ** N_COUNT})")
plt.ylabel(f"fraction of shots giving {answer}")
plt.legend()
plt.show()

## Questions to try

1. What fraction of shots gave the correct answer on each device? How does each compare with random guessing?
2. Set `PHASE = 1/3`, which has no exact 3-digit binary form. In the ideal simulation, which outcome is most likely, and what phase does it correspond to? How spread out is the distribution?
3. Set `N_COUNT = 4` for one more digit of precision. How many two-qubit gates does the circuit need now? Does the hardware still find the answer?
4. If you ran on IBM, compare the two-qubit gate counts for the IBM device and for the qBraid device. The devices have different layouts; how does that show up in the counts?